# BATTERY SOH PREDICTION USING NASA PCOE DATASET

# BUSINESS UNDERSTANDING

## Import Libraries

In [ ]:
import scipy.io
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
import shap

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from scipy.ndimage import uniform_filter1d
from scipy import stats
from scipy.optimize import curve_fit

1. DCT for dQ/dV
2. EIS data use
3. Recursive training 

## Data and Model Configuration

Intialize variables to store data and model configuration information

In [ ]:
DATA_DIR  = Path("./data")
RESULTS_DIR  = Path("./results")
RANDOM_STATE = 42

Initialize Variables to load fixed data objects

In [ ]:
BATTERIES = ["B0005", "B0006", "B0007", "B0018"] #B0036
q_rated = {
    "B0005": 2.00,
    "B0006": 2.00,
    "B0007": 2.00,
    "B0018": 2.00,
    "B0036": 2.00
}
df_qrated = pd.DataFrame.from_dict(q_rated, orient="index", columns=["Q_rated_Ah"])
df_qrated.index.name = "battery"
df_qrated = df_qrated.reset_index()
df_qrated

# DATA UNDERSTANDING

## Data Loading
Load one .mat file and return a flat DataFrame of all discharge timesteps.

Key findings from real files:
- Discharge cycles use Current_load and Voltage_load (not Current_charge/Voltage_charge)
- Capacity is a scalar float per cycle, broadcast across all timesteps
- Each cycle also carries ambient_temperature at the cycle level

In [ ]:
# Data Loading
def load_battery(battery_id: str):
    path = DATA_DIR / f"{battery_id}.mat"
    mat  = scipy.io.loadmat(str(path), simplify_cells=True)
    cycles = mat[battery_id]["cycle"]

    rows = []
    discharge_idx = 0

    for raw_cycle in cycles:
        if raw_cycle["type"] != "discharge":
            continue

        discharge_idx += 1
        d = raw_cycle["data"]
        n = len(d["Time"])
        capacity_Ah = float(d["Capacity"])

        for i in range(n):
            rows.append({
                "battery": battery_id,
                "cycle": discharge_idx,
                "ambient_temp_C": float(raw_cycle["ambient_temperature"]),
                "timestep": i,
                "Time": d["Time"][i],
                "Voltage_measured": d["Voltage_measured"][i],
                "Current_measured": d["Current_measured"][i],
                "Temperature_measured": d["Temperature_measured"][i],
                "Current_load": d["Current_load"][i],
                "Voltage_load": d["Voltage_load"][i],
                "capacity_Ah": capacity_Ah,
            })

    return pd.DataFrame(rows)

def load_all_cycles(battery_id):
    path = DATA_DIR / f"{battery_id}.mat"
    mat = scipy.io.loadmat(str(path), simplify_cells=True)
    cycles = mat[battery_id]["cycle"]

    charge_cycles = []
    discharge_caps = {}
    charge_idx = 0
    discharge_idx = 0

    for raw in cycles:
        if raw["type"] == "charge":
            charge_idx += 1
            charge_cycles.append({
                "battery": battery_id,
                "charge_cycle": charge_idx,
                "ambient_temp": float(raw["ambient_temperature"]),
                "data": raw["data"]
            })

        elif raw["type"] == "discharge":
            discharge_idx += 1
            cap = float(raw["data"]["Capacity"])
            discharge_caps[discharge_idx] = cap

    return charge_cycles, discharge_caps

In [ ]:
# CV phase current decay
def exponential_cv(t, HF4, HF5, HF6):
    return HF4 + HF5 * np.exp(-t / HF6)

In [ ]:
# Merge all battery data into a single DataFrame
raw_dfs = []
for bat in BATTERIES:
    df_bat = load_battery(bat)
    raw_dfs.append(df_bat)
    print(f"{bat}: {df_bat['cycle'].nunique():>3} discharge cycles, "
          f"{len(df_bat):>6,} timesteps")

raw_df = pd.concat(raw_dfs, ignore_index=True)

# Sanity checks
print(raw_df[["Voltage_measured","Current_measured",
              "Temperature_measured","Time","capacity_Ah"]].describe().round(4))

In [ ]:
raw_df.head()

### What is State of Health (SOH) in Battery technology?
Extract the cycle capacity (SOH): Target Variable for Regression Analysis

SOH is a percentage measure of how much usable capacity remains compared to when the battery was new.\
Calculation: SOH(t) = Q_measured(t) / Q_rated \
\
It directly coincides with estimating the Remaining Useful Life (RUL) with respect to defined threshold of capacity fade.

In [ ]:
# Cycle-level capacity table (target variable)
cycle_capacity = raw_df.groupby(["battery","cycle"])["capacity_Ah"].mean().reset_index()
cycle_capacity = pd.merge(cycle_capacity, df_qrated, how="left",on="battery")
cycle_capacity["SoH"] = np.round(cycle_capacity["capacity_Ah"] / cycle_capacity["Q_rated_Ah"],4) 
print(cycle_capacity[["capacity_Ah","SoH"]].sum())
cycle_capacity.head()

In [ ]:
print(cycle_capacity['battery'].unique())
cycle_capacity.describe()

## EXPLORATORY DATA ANALYSIS (EDA)

### EDA questions to contemplate:

1. Does capacity actually degrade?
2. Is the degradation monotonic?
3. Do the raw signals carry the aging signal and are they good indicators of aging?
4. Are the four batteries comparable? What is the extent of Inter-battery Variance for cross-battery testing? 

End of Life Threshold is set at 70% of the total capacity for each specific battery type.

In [ ]:
# Inputs:
#   raw_df: flat time-series, all batteries, discharge only
#   cycle_capacity: one row per cycle with capacity_Ah and SoH

# Consistent color per battery across all plots
BAT_COLORS = {
    "B0005": "#1f77b4",
    "B0006": "#ff7f0e",
    "B0007": "#2ca02c",
    "B0018": "#d62728"
}

# Define End of Life Threshold
EOL_THRESHOLD = 1.4   # Ah: 70%

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

### Plot 1: Capacity fade over cycles (the core degradation signal)
Plot the battery capacity across each discharge cycle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw capacity per cycle
ax = axes[0]
for bat, color in BAT_COLORS.items():
    sub = cycle_capacity[cycle_capacity["battery"] == bat]
    ax.plot(sub["cycle"], sub["capacity_Ah"],
            color=color, alpha=0.55, linewidth=1.0, label=bat)
    # Overlay smoothed trend
    smoothed = uniform_filter1d(sub["capacity_Ah"].values, size=10)
    ax.plot(sub["cycle"], smoothed,
            color=color, linewidth=2.2)

ax.axhline(EOL_THRESHOLD, color="black", linestyle="--",
           linewidth=1.2, label=f"EOL threshold ({EOL_THRESHOLD} Ah)")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Capacity (Ah)")
ax.set_title("Capacity fade over cycles\n(faint = raw, bold = 10-cycle smoothed)")
ax.legend(fontsize=9)

ax = axes[1]
for bat, color in BAT_COLORS.items():
    sub = cycle_capacity[cycle_capacity["battery"] == bat]
    ax.plot(sub["cycle"], sub["SoH"],
            color=color, alpha=0.55, linewidth=1.0, label=bat)
    smoothed = uniform_filter1d(sub["SoH"].values, size=10)
    ax.plot(sub["cycle"], smoothed,
            color=color, linewidth=2.2)

ax.axhline(EOL_THRESHOLD / 2.0, color="black", linestyle="--",
           linewidth=1.2, label="EOL at SoH = 0.70")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("State of Health (SoH = capacity / Q-rated-capacity)")
ax.set_title("Normalised SoH over cycles")
ax.legend(fontsize=9)

plt.suptitle("Plot 1: Core degradation signal", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot1_capacity_fade.png", dpi=150, bbox_inches="tight")
plt.show()

print("Degradation summary")
print(
    cycle_capacity.groupby("battery").agg(
        n_cycles=("cycle", "count"),
        cap_start=("capacity_Ah", "max"),
        cap_end=("capacity_Ah", "min"),
        total_fade=("capacity_Ah", lambda x: x.max() - x.min()),
        fade_pct=("capacity_Ah", lambda x: 100*(x.max()-x.min())/x.max()),
    ).round(3)
)

#### Observation
We observe clear capacity fade from plot 1. Battery B0006 has the mode fade at 43% in 168 cycles. This attends to the core business issue of reduction in battery capacity and the need to plan the battery replacement adequately before the 70% degredation level is reached.

Further, the peaks indicate capacity degradation is non-monotonic. This implies the use of rolling statistics to capture the peaks as indications of further degradation. The peaks are known electrochemical phenomenon called capacity regeneration, caused by lithium redistribution during rest periods between test sessions. They are real signal, not measurement error.

From right pane, we observe degradation range of B0006 is deeper than others. Since B0018 is part of test data and others form the training data, the test distribution is within the range of training distribution. Hence, the inter-battery test/train split does not affect metrics in the current configuration. Battery agnostic features are imperative to capture true patterns.

### Plot 2: Discharge voltage curves across age (the raw signal)
Plot the intr-cycle voltage curves during battery discharge. 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, bat in enumerate(BATTERIES):
    ax = axes[idx]
    sub = raw_df[raw_df["battery"] == bat]
    n_cycles = sub["cycle"].nunique()

    # Sample 8 representative cycles spread across the battery's life
    sample_cycles = np.linspace(1, n_cycles, 8, dtype=int)
    cmap = cm.get_cmap("RdYlGn_r", len(sample_cycles))

    for i, cyc in enumerate(sample_cycles):
        cyc_data = sub[sub["cycle"] == cyc].sort_values("Time")
        ax.plot(
            cyc_data["Time"] / 60,          # in minutes
            cyc_data["Voltage_measured"],
            color=cmap(i), linewidth=1.6,
            label=f"Cycle {cyc}"
        )

    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Voltage (V)")
    ax.set_title(f"{bat} - discharge curves\n(green=early, red=late)")
    ax.legend(fontsize=7.5, loc="lower left")
    ax.set_ylim(2.4, 4.4)

plt.suptitle("Plot 2: Voltage curve evolution across battery life", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot2_voltage_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Discharge duration summary (minutes, per battery)")
duration_df = (
    raw_df.groupby(["battery","cycle"])["Time"]
    .max()
    .reset_index()
    .rename(columns={"Time": "duration_s"})
)
duration_df["duration_min"] = duration_df["duration_s"] / 60
print(
    duration_df.groupby("battery")["duration_min"]
    .agg(["min","max","mean"])
    .round(1)
)

#### Observation
Red Curves (later cycles) have steeper initial drop (higher internal resistance), delayed plateau and reaches lowest voltage earlier. This indicates the Voltage curves clearly demarcates the aging of batteries. This proposes the usage of change in internal resistance, discharge duration, voltage slope and change in quantiles as features as these are distinct to earlier and later cycles.

### Plot 3: Temperature evolution across cycles
Plot the changes in temperature by different cycles of battery.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: temperature curves for one battery across age
ax = axes[0]
bat = "B0005"
sub = raw_df[raw_df["battery"] == bat]
n_cycles = sub["cycle"].nunique()
sample_cycles = np.linspace(1, n_cycles, 8, dtype=int)
cmap = cm.get_cmap("RdYlGn_r", len(sample_cycles))

for i, cyc in enumerate(sample_cycles):
    cyc_data = sub[sub["cycle"] == cyc].sort_values("Time")
    ax.plot(cyc_data["Time"] / 60, cyc_data["Temperature_measured"],
            color=cmap(i), linewidth=1.6, label=f"Cycle {cyc}")

ax.set_xlabel("Time (min)")
ax.set_ylabel("Temperature (°C)")
ax.set_title(f"{bat} - temperature curves across age\n(green=early, red=late)")
ax.legend(fontsize=7.5)

# Right: peak temperature per cycle for all batteries
ax = axes[1]
for bat, color in BAT_COLORS.items():
    sub = raw_df[raw_df["battery"] == bat]
    peak_temp = (
        sub.groupby("cycle")["Temperature_measured"]
        .max()
        .reset_index()
    )
    ax.scatter(peak_temp["cycle"], peak_temp["Temperature_measured"],
               color=color, s=8, alpha=0.6, label=bat)
    smoothed = uniform_filter1d(peak_temp["Temperature_measured"].values, size=10)
    ax.plot(peak_temp["cycle"], smoothed, color=color, linewidth=2.0)

ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Peak temperature (°C)")
ax.set_title("Peak temperature per cycle for all batteries\n"
             "(rising trend = increasing internal heat generation)")
ax.legend(fontsize=9)

plt.suptitle("Plot 3: Temperature as an aging indicator",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot3_temperature.png", dpi=150, bbox_inches="tight")
plt.show()

print("Peak temperature: first 10 vs last 10 cycles per battery")
for bat in BAT_COLORS:
    sub = raw_df[raw_df["battery"] == bat]
    peak = sub.groupby("cycle")["Temperature_measured"].max()
    print(f"{bat}:  early avg = {peak.iloc[:10].mean():.2f}°C  |  "
          f"late avg = {peak.iloc[-10:].mean():.2f}°C  |  "
          f"rise = {peak.iloc[-10:].mean() - peak.iloc[:10].mean():.2f}°C")

#### Observation
Clearly the initial and final temperatures vary according to cycles. This provides temperature as a physical indicative feature for training. The high inter-battery variance proposes to avoid max temperature as a feature. Relatively the rise in temperature will be a better feature for inter-battery model.

### Plot 4: Incremental capacity (dQ/dV) across age
Physics informed metric that demonstrates the rate of change of current with respect to voltage differences across cycles.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, bat in enumerate(BATTERIES):
    ax = axes[idx]
    sub = raw_df[raw_df["battery"] == bat]
    n_cycles = sub["cycle"].nunique()
    sample_cycles = np.linspace(1, n_cycles, 6, dtype=int)
    cmap = cm.get_cmap("RdYlGn_r", len(sample_cycles))

    for i, cyc in enumerate(sample_cycles):
        cyc_data = sub[sub["cycle"] == cyc].sort_values("Voltage_measured")
        V = cyc_data["Voltage_measured"].values
        # Cumulative charge (Ah) via trapezoidal integration of |I| over time
        cyc_time = sub[sub["cycle"] == cyc].sort_values("Time")
        Q_cum = np.cumsum(
            np.abs(cyc_time["Current_measured"].values) *
            np.gradient(cyc_time["Time"].values)
        ) / 3600

        # Map Q onto the voltage-sorted axis
        sort_idx = np.argsort(cyc_time["Voltage_measured"].values)
        V_s = cyc_time["Voltage_measured"].values[sort_idx]
        Q_s = Q_cum[sort_idx]

        # Smooth before differentiating to reduce noise
        Q_smooth = uniform_filter1d(Q_s, size=8)
        with np.errstate(divide="ignore", invalid="ignore"):
            dV = np.gradient(V_s)
            dQdV = np.where(np.abs(dV) > 1e-4,
                            np.gradient(Q_smooth) / dV, 0)

        # Clip outliers for readable plot
        dQdV = np.clip(dQdV, 0, np.percentile(dQdV, 97))

        ax.plot(V_s, dQdV, color=cmap(i), linewidth=1.5,
                alpha=0.8, label=f"Cycle {cyc}")

    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("dQ/dV (Ah/V)")
    ax.set_title(f"{bat} - Incremental capacity (dQ/dV)\n(green=early, red=late)")
    ax.set_xlim(2.5, 4.3)
    ax.legend(fontsize=7.5)

plt.suptitle("Plot 4: dQ/dV: electrochemical fingerprint of degradation",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot4_dqdv.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observation
In early cycles the voltage changes smoothly and gradually across the full range. Hence, dQ/dV stays low and flat with no sharp peak forms. In late cycles the battery reaches the knee region faster, and the voltage plateau becomes more pronounced relative to the total discharge duration.

feature engineering:
Based on understanding from theoretical behavior the peaks are expected in earlier cycles, but the curves show a reversed trend. The peaks present are not truly visible as they are superposed by larger jumps due to scaling. It becomes important to normalise dQdV_peak by discharge duration and compute it only over the low-voltage regionwhere the physical peak actually lives, not over the full voltage range. Voltage position (where the peak occurs on the voltage axis) can be an additional feature.

### Plot 5: Correlation heatmap of candidate features vs target
To eliminate multi-collinearity among independent variables.

In [ ]:
def quick_features(df_bat):
    records = []
    for (bat, cyc), grp in df_bat.groupby(["battery","cycle"]):
        grp = grp.sort_values("Time")
        V = grp["Voltage_measured"].values
        I = grp["Current_measured"].values
        T = grp["Temperature_measured"].values
        t = grp["Time"].values
        dt = np.gradient(t)

        Q_cum = np.trapezoid(np.abs(I), t) / 3600
        E_Wh = np.trapezoid(np.abs(V * I), t) / 3600
        dV_init = np.diff(V[:5]).mean()
        dI_init = np.diff(I[:5]).mean()
        R_int = abs(dV_init / dI_init) if abs(dI_init) > 1e-4 else np.nan

        records.append({
            "battery": bat,
            "cycle": cyc,
            "discharge_dur_s": t[-1] - t[0],
            "voltage_mean": V.mean(),
            "voltage_std": V.std(),
            "voltage_slope": np.polyfit(t, V, 1)[0],
            "temp_max": T.max(),
            "temp_rise": T.max() - T[0],
            "R_int_proxy": R_int,
            "energy_Wh": E_Wh,
            "Q_cum_Ah": Q_cum,
            "capacity_Ah": grp["capacity_Ah"].iloc[0]
        })
    return pd.DataFrame(records)

feat_preview = quick_features(raw_df)

feat_cols = ["discharge_dur_s","voltage_mean","voltage_std","voltage_slope",
             "temp_max","temp_rise","R_int_proxy","energy_Wh","Q_cum_Ah","cycle"]

corr = feat_preview[feat_cols + ["capacity_Ah"]].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True   # show lower triangle only

sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={"size": 9}
)
ax.set_title("Plot 5: Feature correlation matrix\n"
             "(bottom row = correlation with capacity_Ah target)",
             fontweight="bold")
plt.tight_layout()
plt.savefig("./assets/plot5_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Correlation with capacity_Ah")
print(
    corr["capacity_Ah"]
    .drop("capacity_Ah")
    .sort_values(key=abs, ascending=False)
    .round(3)
)

#### Observation
Multi-collinearity causes redundant features for linear modelling. Parameters across features become sensitive to each other it violates the assumption of independent variables in linear baseline. But as ensemble models are less affected all three features can be retained.

### Plot 6: Per-battery degradation comparison 
Validates the cross-battery split strategy for training and test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlay normalised to each battery's own starting capacity
ax = axes[0]
for bat, color in BAT_COLORS.items():
    sub = cycle_capacity[cycle_capacity["battery"] == bat].copy()
    sub["SoH_relative"] = sub["capacity_Ah"] / sub["capacity_Ah"].iloc[0]
    ax.plot(sub["cycle"], sub["SoH_relative"],
            color=color, linewidth=1.8, label=bat, alpha=0.85)

ax.axhline(0.80, color="black", linestyle="--", linewidth=1.0,
           label="80% of initial (common EOL proxy)")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Capacity relative to cycle-1 capacity")
ax.set_title("Relative capacity fade\n(each battery normalised to its own start)")
ax.legend(fontsize=9)

# Right: cycle-by-cycle difference between batteries (B0005 as reference)
ax = axes[1]
ref = (cycle_capacity[cycle_capacity["battery"] == "B0005"]
       .set_index("cycle")["capacity_Ah"])

for bat in ["B0006", "B0007", "B0018", "B0036"]:
    sub = (cycle_capacity[cycle_capacity["battery"] == bat]
           .set_index("cycle")["capacity_Ah"])
    common = ref.index.intersection(sub.index)
    diff = sub.loc[common] - ref.loc[common]
    ax.plot(common, diff, color=BAT_COLORS[bat],
            linewidth=1.5, label=f"{bat} − B0005")

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Discharge cycle")
ax.set_ylabel("Capacity difference vs B0005 (Ah)")
ax.set_title("Inter-battery divergence\n(B0005 as reference)")
ax.legend(fontsize=9)

plt.suptitle("Plot 6: Cross-battery comparison (validates holdout split)",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot6_battery_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Capacity loss per 10 cycles")
for bat in BAT_COLORS:
    sub = cycle_capacity[cycle_capacity["battery"] == bat]
    slope = np.polyfit(sub["cycle"], sub["capacity_Ah"], 1)[0]
    print(f"{bat}:  {slope*10:.4f} Ah / 10 cycles")

#### Observation
This indicates large variance across batteries during early and late phases. Cycle level RMSE is a necessity. Also, battery type encoding features should be avoided such as cycle sequence.

### EDA summary:

1. Degradation is real, measurable and non-linear. Requires complex models that accounts for non-linear relations.
2. The raw signals do carry the aging signal. But requre feature engineering to avoid pitfalls.
3. B0018 is a legitimate holdout. Its degradation trajectory differs from the training batteries in rate and shape, making the cross-battery split a genuine generalisation test rather than an easy one.
4. Key features are highly correlated with the target. Ex: discharge duration, cumulative charge, and energy. Physics-informed features like internal resistance and temp rise provide additional signal shape info.

# DATA PREPARATION

## Feature Engineering

In [ ]:
# reference physical static data 
T_REF = 24.0   # reference ambient temperature (°C)
ALPHA = 0.005  # capacity temperature coefficient for LiCoO2 (Ah/°C)
VMIN_DQDV = 2.7 # voltage window for dQ/dV peak extraction
VMAX_DQDV = 3.7

def extract_cycle_features(grp):
    V  = grp["Voltage_measured"].values
    I  = grp["Current_measured"].values
    T  = grp["Temperature_measured"].values
    t  = grp["Time"].values
    T_amb = grp["ambient_temp_C"].iloc[0]

    # Statistical Features
    discharge_dur_s = t[-1] - t[0]
    voltage_mean = V.mean()
    voltage_std = V.std()
    voltage_slope = np.polyfit(t, V, 1)[0]   # V/s: encodes non-linear relations

    # Voltage at 80% of discharge duration: position-specific snapshot
    idx_80 = int(0.80 * len(t))
    voltage_at_80pct = V[idx_80]

    # temp_rise: [peak-start] to remove inter-battery ambient offset
    temp_rise = T.max() - T[0]

    # Physics-Informed features

    # Internal resistance at early stage: dV/dI at cycle start (first 10 steps)
    dV = np.diff(V[:10]).mean()
    dI = np.diff(I[:10]).mean()
    R_int_proxy = abs(dV / dI) if abs(dI) > 1e-4 else np.nan

    # Energy delivered this cycle (Wh)
    energy_Wh = np.trapezoid(np.abs(V * I), t) / 3600

    # Cumulative charge delivered (Ah)
    Q_cum_Ah = np.trapezoid(np.abs(I), t) / 3600

    # Temperature-compensated capacity
    #  Normalises Q_cum_Ah for ambient temperature drift across cycles
    #  Formula: Q_norm = Q / (1 + alpha * (T_amb - T_ref))
    Q_temp_compensated = Q_cum_Ah / (1 + ALPHA * (T_amb - T_REF))

    # dQ/dV peak features: computed over low-voltage window only
    mask = (V >= VMIN_DQDV) & (V <= VMAX_DQDV)
    dQdV_peak_height  = np.nan
    dQdV_peak_voltage = np.nan

    if mask.sum() > 10:
        V_win = V[mask]
        I_win = I[mask]
        t_win = t[mask]

        # Sort by voltage (ascending) for dQ/dV computation
        sort_idx = np.argsort(V_win)
        V_s = V_win[sort_idx]
        t_s = t_win[sort_idx]
        I_s = I_win[sort_idx]

        # Cumulative charge in this window
        Q_s = np.cumsum(np.abs(I_s) * np.gradient(t_s)) / 3600

        # Smooth Q before differentiating to reduce numerical noise
        Q_smooth = uniform_filter1d(Q_s, size=max(3, len(Q_s) // 15))

        dV_arr = np.gradient(V_s)
        with np.errstate(divide="ignore", invalid="ignore"):
            dQdV = np.where(np.abs(dV_arr) > 1e-5, np.gradient(Q_smooth) / dV_arr, 0)

        # Clip top 2% to suppress remaining noise spikes
        dQdV = np.clip(dQdV, 0, np.percentile(dQdV, 98))

        # Normalise by discharge duration
        dQdV_norm = dQdV / (discharge_dur_s / 3600)

        peak_idx = np.argmax(dQdV_norm)
        dQdV_peak_height  = dQdV_norm[peak_idx]
        dQdV_peak_voltage = V_s[peak_idx]

    return {
        # Statistical
        "discharge_dur_s": discharge_dur_s,
        "voltage_mean": voltage_mean,
        "voltage_std": voltage_std,
        "voltage_slope": voltage_slope,
        "voltage_at_80pct": voltage_at_80pct,
        "temp_rise": temp_rise,
        # Physics-informed
        "R_int_proxy": R_int_proxy,
        "energy_Wh": energy_Wh,
        "Q_cum_Ah": Q_cum_Ah,
        "Q_temp_compensated": Q_temp_compensated,
        "dQdV_peak_height": dQdV_peak_height,
        "dQdV_peak_voltage": dQdV_peak_voltage
    }

#### Health Factors from Charge Cycle
Basis obtained from research paper 

In [ ]:
def extract_hf(charge_cycle):
    d = charge_cycle["data"]

    # Charge cycles use Current_charge and Voltage_charge field names
    V = np.array(d["Voltage_measured"])
    I = np.array(d["Current_measured"])
    t = np.array(d["Time"])

    result = {k: np.nan for k in ["HF1","HF2","HF3","HF4","HF5","HF6"]}

    # HF1: time from V=3.9V to V=4.2V during CC phase
    # CC phase: current is approximately constant (~1.5A)
    idx_39 = np.where(V >= 3.9)[0]
    idx_42 = np.where(V >= 4.2)[0]

    if len(idx_39) > 0 and len(idx_42) > 0:
        t_39 = t[idx_39[0]]
        t_42 = t[idx_42[0]]
        if t_42 > t_39:
            result["HF1"] = t_42 - t_39   # seconds

    # HF2: voltage rise within 600s after reaching 3.9V
    if len(idx_39) > 0:
        t_start = t[idx_39[0]]
        mask_600 = (t >= t_start) & (t <= t_start + 600)
        if mask_600.sum() > 1:
            result["HF2"] = V[mask_600][-1] - V[mask_600][0]

    # HF3: current drop within 900s after entering CV phase
    if len(idx_42) > 0:
        cv_start_idx = idx_42[0]
        t_cv_start   = t[cv_start_idx]
        mask_900 = (t >= t_cv_start) & (t <= t_cv_start + 900)
        if mask_900.sum() > 1:
            I_cv = I[mask_900]
            result["HF3"] = I_cv[0] - I_cv[-1]   # positive = current dropped

    # HF4, HF5, HF6: exponential fit to CV current decay
    # Fit I(t) = HF4 + HF5 * exp(-t/HF6) to the full CV phase
    if len(idx_42) > 0:
        cv_start_idx = idx_42[0]
        V_cv = V[cv_start_idx:]
        I_cv = I[cv_start_idx:]
        t_cv = t[cv_start_idx:] - t[cv_start_idx]   # reset time to 0

        if len(t_cv) > 20 and I_cv.max() > 0.05:
            try:
                # Initial guess: floor~0, amplitude~1.5A, time constant~1000s
                p0 = [0.01, I_cv[0], 1000]
                bounds = ([0, 0, 10], [0.5, 3.0, 20000])
                popt, _ = curve_fit(
                    exponential_cv, t_cv, I_cv,
                    p0=p0, bounds=bounds,
                    maxfev=5000
                )
                result["HF4"] = popt[0]   # asymptotic floor
                result["HF5"] = popt[1]   # initial amplitude
                result["HF6"] = popt[2]   # RC time constant
            except RuntimeError:
                pass   #leave as NaN

    return result


In [ ]:
all_records = []

for bat in BATTERIES:
    charge_cycles, discharge_caps = load_all_cycles(bat)

    # Charge cycle n pairs with discharge cycle n
    n_pairs = min(len(charge_cycles), len(discharge_caps))
    print(f"{bat}: {len(charge_cycles)} charge cycles, "
          f"{len(discharge_caps)} discharge cycles, "
          f"{n_pairs} paired")

    for i in range(n_pairs):
        cyc_idx = i + 1
        hf = extract_hf(charge_cycles[i])
        cap = discharge_caps.get(cyc_idx, np.nan)

        record = {
            "battery": bat,
            "cycle": cyc_idx,
            **hf,
            "capacity_Ah": cap,
            "SoH": cap / 2.0 if not np.isnan(cap) else np.nan,
        }
        all_records.append(record)

hf_df = pd.DataFrame(all_records)

print(f"\nHF table shape: {hf_df.shape}")
print(f"Columns: {hf_df.columns.tolist()}")



### Generate features from function

In [ ]:
# generate
records = []
for (bat, cyc), grp in raw_df.groupby(["battery", "cycle"]):
    grp_sorted = grp.sort_values("Time")
    feat = extract_cycle_features(grp_sorted)
    feat["battery"] = bat
    feat["cycle"] = cyc
    feat["capacity_Ah"] = grp_sorted["capacity_Ah"].iloc[0]
    records.append(feat)

feature_df = pd.DataFrame(records)

# Add rolling statistics
feature_df = feature_df.sort_values(["battery", "cycle"]).reset_index(drop=True)

for bat in feature_df["battery"].unique():
    mask = feature_df["battery"] == bat
    cap  = feature_df.loc[mask, "capacity_Ah"]

    feature_df.loc[mask, "rolling_mean_5"] = cap.shift(1).rolling(window=5, min_periods=1).mean()
    feature_df.loc[mask, "rolling_std_5"] = cap.shift(1).rolling(window=5, min_periods=1).std().fillna(0)

feature_df["rolling_mean_5"] = (
    feature_df.groupby("battery")["rolling_mean_5"]
    .transform(lambda x: x.bfill())
)
feature_df["dQdV_peak_height"] = feature_df["dQdV_peak_height"].clip(lower=0)
feature_df

In [ ]:
print("feature shape")
print(feature_df.shape)

print("\nNulls")
nulls = feature_df.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "No nulls")

print("\nfeature stats")
feat_cols = [
    "discharge_dur_s","voltage_mean","voltage_std","voltage_slope",
    "voltage_at_80pct","temp_rise","R_int_proxy","energy_Wh",
    "Q_cum_Ah","Q_temp_compensated","dQdV_peak_height",
    "dQdV_peak_voltage","rolling_mean_5","rolling_std_5"
]
print(feature_df[feat_cols].describe().round(4))

print("\nsamples by battery")
print(feature_df.groupby("battery").size().rename("n_cycles"))

print(feature_df[["battery","cycle"] + feat_cols + ["capacity_Ah"]].head().to_string(index=False))

### Train/Test Split
The battery data on B0018 is chosen as holdout set while the other batteries serve as the training dataset.

In [ ]:
# Split by battery ID
# B0018 is the holdout: the model has never seen this battery during training.

TRAIN_BATTERIES = ["B0005", "B0006", "B0007", "B0036"]
TEST_BATTERY = "B0018"

train_df = feature_df[feature_df["battery"].isin(TRAIN_BATTERIES)].copy()
test_df = feature_df[feature_df["battery"] == TEST_BATTERY].copy()

X_train = train_df[feat_cols]
y_train = train_df["capacity_Ah"]
X_test = test_df[feat_cols]
y_test = test_df["capacity_Ah"]

print(f"Train : {X_train.shape[0]} cycles  "
      f"| capacity range {y_train.min():.4f}–{y_train.max():.4f} Ah")
print(f"Test  : {X_test.shape[0]} cycles  "
      f"| capacity range {y_test.min():.4f}–{y_test.max():.4f} Ah")
print(f"Features : {len(feat_cols)}")

# MODEL BUILDING

## Initialize Models
Initialize the 3 models as a pipeline to avoid data leakage.

In [ ]:
# Initialize Pipeline
pipelines = {
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0))
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  RandomForestRegressor(n_estimators = 200, max_depth = None, min_samples_leaf = 2, 
                                         random_state = RANDOM_STATE,n_jobs = -1)
        )
    ]),
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  XGBRegressor(n_estimators = 300, learning_rate = 0.05, max_depth = 4, subsample = 0.8, reg_alpha = 0.1,
                                colsample_bytree = 0.8, random_state = RANDOM_STATE, verbosity = 0, n_jobs = -1
        ))
    ]),
}

## Group K-Fold Validation
Train on 2 batteries with one full battery set as hold out.

In [ ]:
# Group Kfold: battery type/ID as folds
# With 3 training batteries this is leave-one-battery-out CV.

groups = train_df["battery"].values
n_splits_var = int(len(groups.unique()))
gkf = GroupKFold(n_splits=n_splits_var)

cv_results = {}
print(f"Cross-validation (GroupKFold, {n_splits_var} folds)")

for name, pipe in pipelines.items():
    cv = cross_validate(
        pipe, X_train, y_train,
        cv = gkf,
        groups = groups,
        scoring = ["neg_root_mean_squared_error", "neg_mean_absolute_error"],
        return_train_score = True,
        n_jobs = -1
    )
    rmse_val = -cv["test_neg_root_mean_squared_error"]
    mae_val = -cv["test_neg_mean_absolute_error"]
    rmse_train = -cv["train_neg_root_mean_squared_error"]

    cv_results[name] = {
        "cv_rmse_mean": rmse_val.mean(),
        "cv_rmse_std": rmse_val.std(),
        "cv_mae_mean": mae_val.mean(),
        "cv_train_rmse": rmse_train.mean(),
    }
    print(f"\n{name}")
    print(f"  Train RMSE (mean): {rmse_train.mean():.4f} Ah")
    print(f"  Val RMSE (mean): {rmse_val.mean():.4f} Ah  ±{rmse_val.std():.4f}")
    print(f"  Val MAE (mean): {mae_val.mean():.4f} Ah")
    gap = rmse_val.mean() - rmse_train.mean()
    print(f"  Generalisation gap (val−train RMSE): {gap:.4f} Ah"
          f"  {'possible overfit' if gap > 0.05 else 'Overfitting'}")
    
    print(f"  Val RMSE by Battery:")
    print(groups.unique())
    print(rmse_val)

## Test on hold out set
Test the models on 4th Battery (B0018) to assess the model performance on unseen data.

In [ ]:
print(f"\nHoldout evaluation on {TEST_BATTERY}")
trained = {}
holdout_results = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    trained[name] = pipe

    y_pred = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)

    holdout_results[name] = {
        "rmse": rmse,
        "mae": mae,
        "y_pred": y_pred
    }
    print(f"{name:>14}  RMSE: {rmse:.4f} Ah   MAE: {mae:.4f} Ah")

### Grid Search with groupK fold to tune XGBoost hyperparameters

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [2, 3, 4],
    "model__learning_rate": [0.05, 0.06],
    "model__min_child_weight": [3, 5, 10],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8],
    "model__reg_alpha": [0.1, 0.5],
    "model__reg_lambda": [0.0],
}

xgb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  XGBRegressor(random_state = RANDOM_STATE, verbosity = 0, n_jobs = -1,
    ))
])

gkf = GroupKFold(n_splits=3)
groups = train_df["battery"].values

## randomly sample 50 combinations to reduce computation
search = GridSearchCV(estimator = xgb_pipe, param_grid = param_grid, 
                            scoring = "neg_root_mean_squared_error", cv = gkf, refit = True,
                            n_jobs = -1, verbose = 1)

search.fit(X_train, y_train, groups=groups)

print("Best parameters")
for param, value in search.best_params_.items():
    print(f"  {param:<30} {value}")

print(f"\nBest CV RMSE : {-search.best_score_:.4f} Ah")

# Evaluate best model on B0018
y_pred_best = search.predict(X_test)
rmse_best = mean_squared_error(y_test, y_pred_best) ** 0.5
mae_best = mean_absolute_error(y_test, y_pred_best)
print(f"B0018 RMSE: {rmse_best:.4f} Ah")
print(f"B0018 MAE: {mae_best:.4f} Ah")

# Check for overfitting
best_pipe = search.best_estimator_
y_pred_train = best_pipe.predict(X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5
print(f"\nTrain RMSE: {rmse_train:.4f} Ah")
print(f"Val RMSE: {-search.best_score_:.4f} Ah")
print(f"Ratio: {(-search.best_score_) / rmse_train:.1f}×  "
      f"{'no overfitting' if (-search.best_score_) / rmse_train < 10 else 'still overfitting'}")

In [ ]:
# Top Combinations
print("\nTop 10 combinations by CV RMSE")
results_df = pd.DataFrame(search.cv_results_)
top10 = (
    results_df
    .sort_values("rank_test_score")
    .head(10)[[
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__learning_rate",
        "param_model__min_child_weight",
        "param_model__subsample",
        "param_model__colsample_bytree",
        "param_model__reg_alpha",
        "param_model__reg_lambda",
        "mean_test_score",
        "std_test_score",
    ]]
    .rename(columns={
        "param_model__n_estimators":"n_est",
        "param_model__max_depth":"depth",
        "param_model__learning_rate":"lr",
        "param_model__min_child_weight":"min_child",
        "param_model__subsample":"subsample",
        "param_model__colsample_bytree":"col_sample",
        "param_model__reg_alpha":"alpha",
        "param_model__reg_lambda":"lambda",
        "mean_test_score":"cv_rmse",
        "std_test_score":"std",
    })
)
top10["cv_rmse"] = (-top10["cv_rmse"]).round(4)
top10["std"] = top10["std"].round(4)
print(top10.to_string(index=False))

xgb_final = search.best_estimator_
xgb_final_pred = y_pred_best

print("\nReady for Bucket 5")
print(f"Final model: XGBoost with best params above")
print(f"B0018 RMSE: {rmse_best:.4f} Ah")

## Generate the metrics to analyse
For each cycle zone: early, mid and late, generate the results. Analysis at 3 zones to minimize the impact of inter-battery variance. 

In [ ]:
print("\nRMSE by life phase on B0018")
cycles_test = test_df["cycle"].values
phases = {
    "Early (1–44)": (cycles_test >= 1)  & (cycles_test <= 44),
    "Mid (45–88)": (cycles_test >= 45) & (cycles_test <= 88),
    "Late (89–132)": (cycles_test >= 89) & (cycles_test <= 132),
}

phase_rows = []
for name in pipelines:
    y_pred = holdout_results[name]["y_pred"]
    row = {"Model": name}
    for phase_label, mask in phases.items():
        rmse_p = mean_squared_error(y_test.values[mask], y_pred[mask]) ** 0.5
        row[phase_label] = round(rmse_p, 4)
    phase_rows.append(row)

phase_df = pd.DataFrame(phase_rows).set_index("Model")
print(phase_df.to_string())

## Model Level Comparison
Compare the RMSE and MAE of the 3 models to analyse the performance.

In [ ]:
print("\nFull model comparison")
rows = []
for name in pipelines:
    rows.append({
        "Model": name,
        "CV RMSE (mean)": round(cv_results[name]["cv_rmse_mean"], 4),
        "CV RMSE (±std)": round(cv_results[name]["cv_rmse_std"],  4),
        "CV MAE": round(cv_results[name]["cv_mae_mean"],   4),
        "B0018 RMSE": round(holdout_results[name]["rmse"],     4),
        "B0018 MAE": round(holdout_results[name]["mae"],      4),
    })
comparison_df = pd.DataFrame(rows).set_index("Model")
print(comparison_df.to_string())

## Residual Plots

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

cycles_test = test_df["cycle"].values
y_true      = y_test.values

for col, name in enumerate(pipelines):
    y_pred    = holdout_results[name]["y_pred"]
    residuals = y_true - y_pred   # positive = under-prediction

    # Top row: residuals vs cycle number
    ax_top = fig.add_subplot(gs[0, col])
    colors = np.where(cycles_test <= 44, "#1f77b4",
             np.where(cycles_test <= 88, "#2ca02c", "#d62728"))
    ax_top.scatter(cycles_test, residuals, c=colors, s=12, alpha=0.7)
    ax_top.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax_top.set_xlabel("Cycle (B0018)")
    ax_top.set_ylabel("Residual (Ah)\ntrue - predicted")
    ax_top.set_title(f"{name}\nResiduals vs cycle")
    # Phase shading
    ax_top.axvspan(1,  44,  alpha=0.04, color="#1f77b4")
    ax_top.axvspan(45, 88,  alpha=0.04, color="#2ca02c")
    ax_top.axvspan(89, 132, alpha=0.04, color="#d62728")

    # Bottom row: predicted vs actual
    ax_bot = fig.add_subplot(gs[1, col])
    ax_bot.scatter(y_true, y_pred, c=colors, s=12, alpha=0.7)
    lims = [min(y_true.min(), y_pred.min()) - 0.02,
            max(y_true.max(), y_pred.max()) + 0.02]
    ax_bot.plot(lims, lims, "k--", linewidth=0.8, label="Perfect prediction")
    ax_bot.set_xlabel("True capacity (Ah)")
    ax_bot.set_ylabel("Predicted capacity (Ah)")
    ax_bot.set_title(f"{name}\nPredicted vs actual")
    ax_bot.set_xlim(lims); ax_bot.set_ylim(lims)

# Legend for life phase colours
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#1f77b4", label="Early (1–44)"),
    Patch(facecolor="#2ca02c", label="Mid (45–88)"),
    Patch(facecolor="#d62728", label="Late (89–132)"),
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.suptitle("Bucket 4: Residual analysis on B0018 holdout\n"
             "(blue=early, green=mid, red=late cycles)",
             fontweight="bold", y=1.02)
plt.savefig("./assets/plot7_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

#### Observation:
1. Ridge's early-cycle bias is interpretable and honest. It reflects a real inter-battery difference, not a model failure. The correct statement in your write-up is: "Ridge over-predicts B0018 capacity in early cycles because B0018's initial capacity (1.855 Ah) is lower than the training battery mean. A model without battery-specific calibration will anchor to the training distribution."
2. Ensemble mid-cycle bias reveals overfit pattern matching. RF and XGBoost learned the specific shape of training battery degradation curves including their recovery bumps. When B0018's recovery pattern differs, the models mis-predict. This is the practical consequence of the 41× train/val ratio identified in Bucket 4.
3. All models are operationally reliable in the late-life region. Red points cluster tightly on the diagonal for all three models — the EOL region (1.34–1.45 Ah) is predicted accurately. For a real BMS application this is the region that matters most, so all three models would be deployable for end-of-life detection despite their mid-life errors.

# MODEL EVALUATION

In [ ]:
# Cycle Level Metrics
cycles_test = test_df["cycle"].values
y_true = y_test.values

phases = {
    "Early (1–44)":(cycles_test >= 1)  & (cycles_test <= 44),
    "Mid (45–88)":(cycles_test >= 45) & (cycles_test <= 88),
    "Late (89–132)":(cycles_test >= 89) & (cycles_test <= 132)
}

# Summary
print("Holdout metrics (B0018)")
full_rows = []
for name in ["Ridge", "RandomForest", "XGBoost"]:
    y_pred = holdout_results[name]["y_pred"]
    resid  = y_true - y_pred
    full_rows.append({
        "Model": name,
        "RMSE": round(mean_squared_error(y_true, y_pred)**0.5, 4),
        "MAE": round(mean_absolute_error(y_true, y_pred), 4),
        "R²": round(r2_score(y_true, y_pred), 5),
        "Bias(mean)": round(resid.mean(), 5),   # systematic over/under prediction
        "Resid std": round(resid.std(), 5),     # spread of errors
        "Max |err|": round(np.abs(resid).max(), 4),
    })
full_df = pd.DataFrame(full_rows).set_index("Model")
print(full_df.to_string())

In [ ]:
# Phase Level Metrics

print("\nPhase-level RMSE and bias (B0018)")
phase_rows = []
for name in ["Ridge", "RandomForest", "XGBoost"]:
    y_pred = holdout_results[name]["y_pred"]
    row    = {"Model": name}
    for label, mask in phases.items():
        resid_p = y_true[mask] - y_pred[mask]
        row[f"{label} RMSE"] = round(mean_squared_error(y_true[mask], y_pred[mask])**0.5, 4)
        row[f"{label} bias"] = round(resid_p.mean(), 4)
    phase_rows.append(row)

phase_df = pd.DataFrame(phase_rows).set_index("Model")
print(phase_df.to_string())

In [ ]:
# Residual Normality check

print("\nResidual distribution (Shapiro-Wilk normality test)")
for name in ["Ridge", "RandomForest", "XGBoost"]:
    resid = y_true - holdout_results[name]["y_pred"]
    stat, p = stats.shapiro(resid)
    skew = stats.skew(resid)
    kurt = stats.kurtosis(resid)
    print(f"{name:>14}  W={stat:.4f}  p={p:.4f}  "
          f"skew={skew:.3f}  kurtosis={kurt:.3f}  "
          f"{'normal' if p > 0.05 else 'non-normal'}")

In [ ]:
# Train/Val/Test Summary

print("\nThree-way comparison: train / CV val / B0018 holdout")
summary_rows = []
for name in ["Ridge", "RandomForest", "XGBoost"]:
    train_pred = trained[name].predict(X_train)
    train_rmse = mean_squared_error(y_train, train_pred)**0.5
    val_rmse   = cv_results[name]["cv_rmse_mean"]
    test_rmse  = holdout_results[name]["rmse"]
    summary_rows.append({
        "Model": name,
        "Train RMSE": round(train_rmse, 4),
        "CV Val RMSE": round(val_rmse, 4),
        "B0018 RMSE": round(test_rmse, 4),
        "Val/Train ratio": round(val_rmse / train_rmse, 1),
        "Test/Val ratio": round(test_rmse / val_rmse, 2),
    })
summary_df = pd.DataFrame(summary_rows).set_index("Model")
print(summary_df.to_string())

In [ ]:
# Error Distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name in zip(axes, ["Ridge", "RandomForest", "XGBoost"]):
    resid  = y_true - holdout_results[name]["y_pred"]
    colors = np.where(cycles_test <= 44,  "#1f77b4",
             np.where(cycles_test <= 88,  "#2ca02c", "#d62728"))

    ax.hist(resid, bins=25, color="steelblue", alpha=0.6, edgecolor="white")
    ax.axvline(0, color="black", linewidth=1.0, linestyle="--")
    ax.axvline(resid.mean(),color="#d62728",linewidth=1.5,
               linestyle="-", label=f"Mean={resid.mean():.4f} Ah")
    ax.set_xlabel("Residual (Ah)")
    ax.set_ylabel("Count")
    ax.set_title(f"{name}\nResidual distribution")
    ax.legend(fontsize=9)

plt.suptitle("Bucket 5: Residual distributions on B0018",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("./assets/plot8_residual_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

#### Final Model
Decision criteria (in priority order):
  1. Explainability: SHAP must explain genuine degradation patterns, not memorised training artefacts
  2. Generalisation: low train→val overfitting ratio
  3. Holdout RMSE: performance on unseen battery

Ridge:
  + Lowest B0018 RMSE (0.0051 Ah), lowest val/train ratio (4.8×)
  + Residuals show interpretable systematic bias (inter-battery offset)
  - Coefficients already interpretable as a result SHAP adds limited insight
  - Cannot capture non-linear interactions between features

XGBoost (tuned):
  + Native TreeSHAP: exact, fast, captures interaction effects
  + Non-linear: can model the knee point and recovery bump dynamics
  - Train→val ratio 41× before tuning — must verify tuned ratio < 10×
  - SHAP explanations only valid if overfitting is resolved
""")

# MODEL EXPLAINABILITY

#### SHAP Explainability
fit linear and tree-based shap explainer to analyse the feature influence.

In [ ]:
# shap.initjs()   

def get_model_and_data(pipeline, X):
    X_transformed = pipeline[:-1].transform(X)
    model = pipeline[-1]              
    X_df = pd.DataFrame(X_transformed, columns=feat_cols)
    return model, X_df

Linear Explainer

In [ ]:
print("Computing Ridge SHAP values...")
ridge_model, X_train_scaled = get_model_and_data(trained["Ridge"], X_train)
_, X_test_scaled = get_model_and_data(trained["Ridge"], X_test)

ridge_explainer = shap.LinearExplainer(ridge_model, X_train_scaled)
ridge_shap_train = ridge_explainer(X_train_scaled)
ridge_shap_test = ridge_explainer(X_test_scaled)

Tree Explainer

In [ ]:
print("Computing XGBoost SHAP values...")
xgb_model, X_train_xgb = get_model_and_data(trained["XGBoost"], X_train)
_, X_test_xgb = get_model_and_data(trained["XGBoost"], X_test)

xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_train = xgb_explainer(X_train_xgb)
xgb_shap_test = xgb_explainer(X_test_xgb)

Summary Plots

In [ ]:
# The beeswarm plot shows:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plt.sca(axes[0])
shap.summary_plot(
    ridge_shap_train,
    X_train_scaled,
    plot_type = "dot",
    show = False,
    title = ""
)
axes[0].set_title("Ridge: SHAP summary (training set)\nLinearExplainer", fontweight="bold")

plt.sca(axes[1])
shap.summary_plot(
    xgb_shap_train,
    X_train_xgb,
    plot_type = "dot",
    show = False,
    title = ""
)
axes[1].set_title("XGBoost: SHAP summary (training set)\nTreeExplainer", fontweight="bold")

plt.suptitle("Plot 9: Global feature importance via SHAP", fontweight="bold", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig("./assets/plot9_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

Bar Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, shap_vals, label in zip(
    axes,
    [ridge_shap_train, xgb_shap_train],
    ["Ridge (LinearExplainer)", "XGBoost (TreeExplainer)"]
):
    mean_abs = np.abs(shap_vals.values).mean(axis=0)
    importance = pd.Series(mean_abs, index=feat_cols).sort_values(ascending=True)

    colors = ["#C8102E" if "dQdV" in f or "R_int" in f or
              "energy" in f or "Q_temp" in f or "Q_cum" in f
              else "#003366"
              for f in importance.index]

    importance.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.set_xlabel("Mean |SHAP value| (Ah)")
    ax.set_title(f"{label}\nMean absolute SHAP: training set", fontweight="bold")
    ax.axvline(0, color="black", linewidth=0.5)

    # Legend
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="#C8102E", label="Physics-informed"),
        Patch(color="#003366", label="Statistical"),
    ], fontsize=9, loc="lower right")

plt.suptitle("Plot 10: Feature importance: physics-informed vs statistical",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot10_shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()

Dependence Plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Features to examine — top 3 by importance, expected from EDA
dep_features  = ["discharge_dur_s", "Q_cum_Ah", "R_int_proxy"]
# Interaction features — what modulates each relationship
inter_features = ["rolling_mean_5", "voltage_mean", "temp_rise"]

for row, (shap_vals, X_data, label) in enumerate([
    (xgb_shap_train, X_train_xgb, "XGBoost"),
    (ridge_shap_train, X_train_scaled, "Ridge"),
]):
    for col, (feat, inter) in enumerate(zip(dep_features, inter_features)):
        ax = axes[row, col]
        shap.dependence_plot(
            feat,
            shap_vals.values,
            X_data,
            interaction_index = inter,
            ax = ax,
            show = False,
        )
        ax.set_title(f"{label}: {feat}\n(color = {inter})", fontsize=10)

plt.suptitle("Plot 11: SHAP dependence plots\n"
             "(how each feature's value maps to its prediction contribution)",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot11_shap_dependence.png", dpi=150, bbox_inches="tight")
plt.show()

Force Plot for XGBoost

In [ ]:
target_cycles = {
    "Cycle 1 (healthy)": 0,
    "Cycle 66 (mid-life)": 65,
    "Cycle 132 (EOL)": 131,
}

fig, axes = plt.subplots(len(target_cycles), 1, figsize=(16, 9))

for ax, (label, idx) in zip(axes, target_cycles.items()):
    shap_row = xgb_shap_test[idx]
    base_value = xgb_shap_test.base_values[idx]
    pred_value = base_value + xgb_shap_test.values[idx].sum()

    # Manual waterfall — more readable in static output than force plot
    contributions = pd.Series(
        xgb_shap_test.values[idx], index=feat_cols
    ).sort_values(key=abs, ascending=False).head(8)

    colors = ["#C8102E" if v > 0 else "#003366"
              for v in contributions.values]

    ax.barh(contributions.index, contributions.values,
            color=colors, edgecolor="white", height=0.6)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("SHAP value (Ah)")
    ax.set_title(
        f"{label}  |  Baseline: {base_value:.4f} Ah  to  "
        f"Predicted: {pred_value:.4f} Ah  |  "
        f"True: {y_test.values[idx]:.4f} Ah",
        fontsize=10, fontweight="bold"
    )
    ax.text(0.99, 0.05, "Red = pushes capacity UP\nBlue = pushes capacity DOWN",
            transform=ax.transAxes, ha="right", fontsize=8,
            color="gray")

plt.suptitle("Plot 12: XGBoost SHAP waterfall: single-cycle explanations\n"
             "(top 8 features by |SHAP| for B0018 cycles 1, 66, 132)",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./assets/plot12_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

Cross-Model Importance

In [ ]:
print("Computing RF permutation importance...")
rf_perm = permutation_importance(
    trained["RandomForest"], X_test, y_test,
    n_repeats=30, random_state=42, n_jobs=-1
)

ridge_imp = pd.Series(
    np.abs(ridge_shap_train.values).mean(axis=0),
    index=feat_cols
)
xgb_imp = pd.Series(
    np.abs(xgb_shap_train.values).mean(axis=0),
    index=feat_cols
)
rf_imp = pd.Series(
    rf_perm.importances_mean,
    index=feat_cols
)

importance_df = pd.DataFrame({
    "Ridge SHAP": ridge_imp,
    "XGBoost SHAP": xgb_imp,
    "RF Permutation": rf_imp,
}).sort_values("XGBoost SHAP", ascending=False)

# Add feature type label
physics = {"R_int_proxy","energy_Wh","Q_cum_Ah","Q_temp_compensated",
           "dQdV_peak_height","dQdV_peak_voltage"}
importance_df["Type"] = ["Physics" if f in physics else "Statistical"
                         for f in importance_df.index]

print("\nCross-model feature importance (mean |SHAP| or permutation)")
print(importance_df.round(5).to_string())

# DEPLOYMENT